# Modelica to PyPowSyBl: Dynamic Simulation Toolkit

### Industrialized Workflow
1. **Create Network:** Programmatically build the static loadflow structure via PyPowSyBl.
2. **Initialize Loadflow:** Define load and generation profiles.
3. **Run Loadflow:** Execute the base AC Loadflow using OpenLF.
4. **Local Model Compilation:** Explicitly compile a Modelica preassembled model (`GeneratorPV`) into a `.so` library using the local `myEnvDynawo.sh` developer script.
5. **Physical & Initialization Parameters:** Build the `.PAR` files using physical parameters and Loadflow initialization references.
6. **Execute Dynamic Simulation:** Run Dynawo time-domain simulation for events.

## Step 0: Environment Setup (Dynawo Install & Config)

We set up the configuration directory so PyPowSyBl can locate the Dynawo installation and the compiled libraries.

In [ ]:
!curl -L $(curl -s -L -X GET https://api.github.com/repos/dynawo/dynawo/releases/latest | grep "Dynawo_Linux" | grep url | cut -d '"' -f 4) -o Dynawo_Linux_latest.zip
!unzip -o Dynawo_Linux_latest.zip > /dev/null 2>&1
!rm Dynawo_Linux_latest.zip 
!./dynawo/dynawo.sh help

In [ ]:
CONFIG_DIR="$HOME/.itools"
!mkdir -p "$CONFIG_DIR"
![ -e "$CONFIG_DIR/config.yml" ] && mv "$CONFIG_DIR/config.yml" "$CONFIG_DIR/config_$(date +%Y%m%d_%H%M%S).yml"
!cp ./getting_started_data/config.yml "$CONFIG_DIR/config.yml"
!sed -i "s|WORKING_DIR|$(pwd)|g" "$CONFIG_DIR/config.yml"

print("\nConfiguration successfully mirrored and saved at ~/.itools/config.yml:")
!cat "$CONFIG_DIR/config.yml"

In [ ]:
import os
import numpy as np
import pandas as pd
import pypowsybl as pp
import pypowsybl.dynamic as dyn
import pypowsybl.report as rp
import matplotlib.pyplot as plt
from IPython.display import SVG, display, HTML

# Ensure plots are displayed inline within the Jupyter environment
%matplotlib inline

print(f"PyPowSyBl version: {pp.__version__}")

current_dir = os.getcwd()
dynawo_dir = os.path.join(current_dir, "dynawo")
data_dir = os.path.join(current_dir, "getting_started_data")

## Step 1 & 2: Create Network and Initialize Loadflow Profiles

We construct the 2-bus network structure programmatically using the provided utility function.

In [ ]:
def create_tutorial_network() -> pp.network.Network:
    """
    Constructs a basic 2-bus static network.
    """
    net = pp.network.load('getting_started_data/Base_Case.xiidm')

    net.create_substations(id=["SUB_GEN", "SUB_LOAD"], name=["Generator Substation", "Load Substation"])
    net.create_voltage_levels(id=["VL_GEN", "VL_LOAD"],
                              name=["VL_GEN", "VL_LOAD"],
                              substation_id=["SUB_GEN", "SUB_LOAD"],
                              topology_kind=["BUS_BREAKER", "BUS_BREAKER"],
                              nominal_v=[400.0, 400.0])

    net.create_buses(id=["BUS_GEN", "BUS_LOAD"],
                     name=["BUS_GEN", "BUS_LOAD"],
                     voltage_level_id=["VL_GEN", "VL_LOAD"])

    net.create_generators(id="GEN_1",
                          name="GEN_1",
                          voltage_level_id="VL_GEN",
                          bus_id="BUS_GEN",
                          target_p=150.0,
                          target_v=400.0,
                          min_p=0.0,
                          max_p=300.0,
                          voltage_regulator_on=True)

    net.create_loads(id="LOAD_1",
                     name="LOAD_1",
                     voltage_level_id="VL_LOAD",
                     bus_id="BUS_LOAD",
                     p0=150.0,
                     q0=50.0)

    net.create_lines(id="LINE_1",
                     name="LINE_1",
                     voltage_level1_id="VL_GEN", bus1_id="BUS_GEN",
                     voltage_level2_id="VL_LOAD", bus2_id="BUS_LOAD",
                     r=1.5, x=15.0, b1=0.0005, b2=0.0005)

    return net

print("Creating network structure...")
network = create_tutorial_network()


## Step 3: Run Loadflow

Execute the base Loadflow using PyPowSyBl's OpenLF.

In [ ]:
print("Running Loadflow (OpenLF)...")
lf_parameters = pp.loadflow.Parameters()
lf_results = pp.loadflow.run_ac(network, parameters=lf_parameters)

print(f"Loadflow Status: {lf_results[0].status.name}")
display(network.get_buses()[["name", "v_mag", "v_angle"]])

## Step 4: Local Model Compilation

We use the local developer script `myEnvDynawo.sh` to compile the standard `GeneratorPV` model into a shared object (`.so`). This relies on the internal OMC engine properly parsing the Modelica code.

In [ ]:
print("Compiling Modelica preassembled model (GeneratorPV) via OpenModelica...")
!./myEnvDynawo.sh compileModelicaModel --model Dynawo.Electrical.Machines.GeneratorPV --lib GeneratorPV.so
print("Compilation finished. Library GeneratorPV.so is ready for linking.")

We map the static component `GEN_1` to our newly compiled `GeneratorPV` model. Ensure the dynamic linker looks for the `.so` extension as defined.

In [ ]:
print("Mapping static elements to precompiled dynamic Modelica classes...")
mapping = dyn.ModelMapping()

df_gen = pd.DataFrame([{
    "static_id": "GEN_1",
    "parameter_set_id": "GEN_1",
    "model_name": "GeneratorPV"
}]).set_index("static_id", drop=False)
mapping.add_synchronous_generator(df_gen)
print("Mapping complete.")

## Step 5: Physical Parameters and Loadflow Initialization

Construct the `.PAR` files configuring the physical properties for `GeneratorPV` and injecting `<reference>` pointers to retrieve $P$, $Q$, $V$, and $\theta$ from the Loadflow solution.

In [ ]:
print("Generating dynamic physical parameters and LF initialization mapping...")

xml_base_case = '''<?xml version="1.0" encoding="UTF-8"?>
<parametersSet xmlns="http://www.rte-france.com/dynawo">
    <set id="GEN_1">
        <par type="DOUBLE" name="generator_H" value="3"/>
        <par type="DOUBLE" name="generator_PNomAlt" value="4275"/>
        <par type="DOUBLE" name="generator_PNomTurb" value="4275"/>
        <par type="DOUBLE" name="generator_SNom" value="4500"/>
        <par type="DOUBLE" name="generator_Tpd0" value="5"/>
        <par type="DOUBLE" name="generator_UNom" value="400"/>
        <par type="DOUBLE" name="generator_XdPu" value="1.1"/>
        <par type="DOUBLE" name="generator_XqPu" value="0.7"/>
        <par type="BOOL" name="generator_UseApproximation" value="true"/>
        
        <reference type="DOUBLE" name="generator_P0Pu" origData="IIDM" origName="p_pu"/>
        <reference type="DOUBLE" name="generator_Q0Pu" origData="IIDM" origName="q_pu"/>
        <reference type="DOUBLE" name="generator_U0Pu" origData="IIDM" origName="v_pu"/>
        <reference type="DOUBLE" name="generator_UPhase0" origData="IIDM" origName="angle"/>
    </set>
</parametersSet>
'''

xml_network = '''<?xml version="1.0" encoding="UTF-8"?>
<parametersSet xmlns="http://www.rte-france.com/dynawo">
    <set id="Network">
        <par type="DOUBLE" name="capacitor_no_reclosing_delay" value="300.0"/>
        <par type="DOUBLE" name="dangling_line_currentLimit_maxTimeOperation" value="90.0"/>
        <par type="DOUBLE" name="line_currentLimit_maxTimeOperation" value="90.0"/>
        <par type="DOUBLE" name="load_Tp" value="90.0"/>
        <par type="DOUBLE" name="load_Tq" value="90.0"/>
        <par type="DOUBLE" name="load_alpha" value="1.0"/>
        <par type="DOUBLE" name="load_beta" value="2.0"/>
        <par type="BOOL" name="load_isControllable" value="false"/>
        <par type="BOOL" name="load_isRestorative" value="false"/>
        <par type="BOOL" name="BUS_LOAD_hasShortCircuitCapabilities" value="true"/>
    </set>
</parametersSet>
'''

data_dir = "getting_started_data"
os.makedirs(data_dir, exist_ok=True)
with open(os.path.join(data_dir, "Base_Case.par"), "w") as f: f.write(xml_base_case)
with open(os.path.join(data_dir, "Network.par"), "w") as f: f.write(xml_network)
print("Parameters saved.")

## Step 6: Execute Dynamic Simulation

Introduce a temporal fault (short-circuit) at the load bus and track the terminal voltages of the system.

In [ ]:
print("Configuring short-circuit event...")
events = dyn.EventMapping()
events.add_node_fault(static_id="BUS_LOAD", start_time=2.0, fault_time=0.1, r_pu=0.0, x_pu=0.0001)

outputs = dyn.OutputVariableMapping()
outputs.add_standard_model_curves("BUS_GEN", "U_value")
outputs.add_standard_model_curves("BUS_LOAD", "U_value")

sim_parameters = dyn.Parameters(start_time=0.0, stop_time=10.0)
simulation = dyn.Simulation()

print("Running dynamic simulation...")
results = simulation.run(
    network,
    model_mapping=mapping,
    event_mapping=events,
    timeseries_mapping=outputs,
    parameters=sim_parameters
)

if results.status().name == 'SUCCESS':
    print("Simulation Successful!")
    curves = results.curves()
    plt.figure(figsize=(10, 5))
    plt.plot(curves.index, curves["NETWORK_BUS_LOAD_U_value"], label="Load Bus Voltage (PU)")
    plt.plot(curves.index, curves["NETWORK_BUS_GEN_U_value"], label="Generator Bus Voltage (PU)", linestyle="--")
    plt.title("Transient Response: Bus Voltages")
    plt.xlabel("Time (s)")
    plt.ylabel("Voltage (PU)")
    plt.grid(True)
    plt.legend()
    plt.show()
else:
    print(f"Simulation failed: {results.status_text()}")